# MCXN9xx Secure Binary 3.1

This notebook describes how to build a Secure Binary 3.1 (SB3.1) file for the MCXN9xx device family (e.g. `mcxn946`/`mcxn947`) using the SPSDK *nxpimage* application and how to load it to the device with *blhost*.

The SB3.1 file format is encrypted and digitally signed. It can be used for a secure firmware update in the field: the running (already trusted) code loads the new SB3.1 container, the ROM/bootloader authenticates it against the Root of Trust and decrypts it using the device's Part Common Key (SB3KDK), and only then commits it to memory. This extends the chain of trust from the currently executing code to the new image and keeps the payload confidential while in transit.

<div class="alert alert-block alert-warning">
<b>WARNING:</b> The keys used in this example are meant only for demonstration purposes and must never be used in production.
</div>

## 1. Prerequisites

- SPSDK is needed with examples extension. `pip install spsdk[examples]` (Please refer to the [installation](../../_knowledge_base/installation_guide.rst) documentation.)
- This example uses an MCXN9xx board (e.g. FRDM-MCXN947) connected via USB (ISP mode) or UART.
- The device is provisioned (Root of Trust and Part Common Key) directly in this notebook using DEVHSM, so no other example has to be run before. The provisioning is the same as in the [DEVHSM example](../../devhsm/mcxn9xx_devhsm/mcxn9xx_devhsm.ipynb).

<div class="alert alert-block alert-info">
<b>NOTE:</b> The CUST_MK_SK key blob is stored in the CMPA page (<code>CUST_MK_SK_KEY_BLOB</code>). Any later CMPA write (e.g. <code>pfr write -t cmpa</code> in other MCXN9xx examples) that does not contain this key blob removes the key from the device and the SB3.1 file is then rejected with <code>1 (0x1) Fail</code>. In such a case, run the provisioning step of this notebook again.
</div>

<img src="../../_data/img/boards/frdm-mcxn947.png" alt="frdm-mcxn947" width="400">

In [1]:
# Initialization cell
from spsdk.utils.jupyter_utils import YamlDiffWidget

# This env variable sets colored logger output to STDOUT
%env JUPYTER_SPSDK=1
# Set a magic for command execution and echo
%alias execute echo %l && %l
%alias_magic ! execute

env: JUPYTER_SPSDK=1
Created `%!` as an alias for `%execute`.


In [2]:
FAMILY = "mcxn946"  # change this to your MCXN9xx family (e.g. mcxn947)
WORKSPACE = "workspace/"  # change this to path to your workspace
INPUTS = "inputs/"  # input configurations and binaries
KEYS = "../../_data/keys/ecc256/"  # shared example keys
VERBOSITY = (
    ""  # verbosity of commands, might be -v or -vv for debug or blank for no additional info
)

## 2. Root of Trust Keys

To build a signed SB3.1 file a Root of Trust (RoT) must be established. The hash of the root key table (RKTH) is fused into the device and cannot be changed afterwards, so the keys are fixed for the lifetime of the device.

In this example we reuse the shared ECC-256 example key-pairs from the `_data/keys/ecc256` folder. In a real scenario you would generate your own key-pairs using the *nxpcrypto* application (refer to [How-to-get-keys-using-nxpcrypto](../../crypto/keys.ipynb)).

## 3. Certificate Block v2.1

The certificate block holds the Root of Trust information (the root key table) and is referenced by the SB3.1 configuration. In this example a ready-made certificate block configuration is provided in the inputs folder (see [cert_block_mcxn9xx.yaml](./inputs/cert_block_mcxn9xx.yaml)). To learn how to create a certificate block from scratch, check the dedicated notebook (see [How-to-get-cert-block](../../certificate_block/cert_block_v2_1.ipynb)).

## 4. Part Common Key (SB3KDK)

The Part Common Key (PCK, also referred to as SB3KDK) is a 256-bit customer key used to derive the encryption keys for the individual SB3.1 blocks. The same key must be provisioned into the device so the ROM can decrypt the container. In this example the key is stored as a plain hex string in [pck.txt](./inputs/pck.txt) and the very same file is used both for the SB3.1 encryption and for the device provisioning (chapter 8), so the keys always match.

The `kdkAccessRights` value in the SB3.1 configuration is part of the key derivation and must match the access rights the ROM uses for the provisioned key. For MCXN9xx with the CUST_MK_SK provisioned by DEVHSM, use `kdkAccessRights: 0`; other values produce a valid-looking SB3.1 file that the device rejects.

## 5. Prepare the SB3.1 configuration file

The *nxpimage* tool generates the SB3.1 file according to a YAML configuration file. A template can be created with the `nxpimage sb31 get-template` command and then customized.

Below we compare the generated template with the ready-made user configuration ([sb31_config.yaml](./inputs/sb31_config.yaml)) to highlight the changes we made. In this example the SB3.1 simply loads a small demo payload into RAM; in practice you would erase the flash and load your application image.

In [3]:
# Get difference of template and user YAML configuration
YamlDiffWidget("inputs/mcxn9xx_sb31.diffc").html

nxpimage sb31 get-template -f mcxn946 -o workspace/sb31_template.yaml --force 
Creating workspace/sb31_template.yaml template file.


## 6. SB3.1 generation

With the certificate block and the Part Common Key in place, we can generate the SB3.1 file. The certificate block referenced in the configuration is built automatically as part of the export.

In [4]:
import os

SB31_CONFIG = INPUTS + "sb31_config.yaml"
%! nxpimage $VERBOSITY sb31 export -c $SB31_CONFIG
assert os.path.exists(WORKSPACE + "mcxn9xx_sb31.sb3")

nxpimage  sb31 export -c inputs/sb31_config.yaml 
RKTH: e2cca7cf09a45d2f1942969fda1c68ecaad78fad416d143292dad2f618291ddd
PFR CMPA configuration has been stored in: workspace/sb31_pfr_cmpa.yaml
Success. (Secure binary 3.1: workspace/mcxn9xx_sb31.sb3 created.)


## 7. Device preparation

Now it is time to prepare the device. Put the board into In-System Programming (ISP) mode, then use *nxpdevscan* to check that the device is detected by the PC.

There are several ways to enter ISP mode:
- Force ISP mode by HW pin (refer to the board user manual),
- Force ISP mode with a debug mailbox command: `nxpdebugmbox -i <interface> cmd -f <family> ispmode -m 0`,
- Native fall-down when the chip has no valid application to boot.

In [5]:
# check if the device is connected and detected by the PC
%! nxpdevscan

nxpdevscan 
Scanning USB devices...
╔══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                                        Connected NXP USB Devices  (1 device)                                         ║
╠══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╣
║  VID       : 0x1fc9                                                                                                  ║
║  PID       : 0x014f                                                                                                  ║
║  Product   : USB COMPOSITE DEVICE                                                                                    ║
║  Name      : kw43b43zc3 | mcxn947 | kw43l43z92 | mcxn556s | kw43l43z93 | mcxw706c | mcxn527t | mcxn557s | mcxw706a | ║
║  Path      : HID\VID_1FC9&PID_014F\6&159A9FAD&1&0000                                                               

In [6]:
# Select the interface to the target. USB (default MCXN9xx ISP VID:PID) or UART.
USB = "-u 0x1fc9,0x014f"  # change to your device, or use e.g. "-p com10" for UART

## 8. Device provisioning (DEVHSM)

The device accepts the SB3.1 file only if it has the matching Root of Trust (RKTH in CMPA) and Part Common Key (CUST_MK_SK). Both are provisioned here in a secure way with Device HSM (DEVHSM), the same flow as in the [DEVHSM example](../../devhsm/mcxn9xx_devhsm/mcxn9xx_devhsm.ipynb):

1. Generate the CMPA (with the RKTH taken from the same certificate block as used by the SB3.1) and CFPA binaries with *pfr*.
2. Generate the provisioning SB file with *nxpdevhsm*. The device encrypts and signs the SB blocks on-chip, and the CUST_MK_SK from [pck.txt](./inputs/pck.txt) is wrapped and injected automatically (see [devhsm_config.yaml](./inputs/devhsm_config.yaml)).
3. Load the provisioning SB file with *blhost* and reset the device so the new configuration takes effect.

<div class="alert alert-block alert-warning">
<b>WARNING:</b> This step overwrites the CMPA and CFPA of the device. The life cycle is not advanced, so the device stays re-programmable.
</div>

In [7]:
import os

CMPA_CONFIG = INPUTS + "cmpa_mcxn9xx.yaml"
CFPA_CONFIG = INPUTS + "cfpa_mcxn9xx.yaml"
CERT_BLOCK_CONFIG = INPUTS + "cert_block_mcxn9xx.yaml"
CMPA_BIN = WORKSPACE + "cmpa.bin"
CFPA_BIN = WORKSPACE + "cfpa.bin"

# CMPA with the RKTH of the same certificate block as used by the SB3.1 file
%! pfr $VERBOSITY export -c $CMPA_CONFIG -e $CERT_BLOCK_CONFIG -o $CMPA_BIN
assert os.path.exists(CMPA_BIN)
%! pfr $VERBOSITY export -c $CFPA_CONFIG -o $CFPA_BIN
assert os.path.exists(CFPA_BIN)

pfr  export -c inputs/cmpa_mcxn9xx.yaml -e inputs/cert_block_mcxn9xx.yaml -o workspace/cmpa.bin 
Success. (CMPA binary has been generated and stored into workspace/cmpa.bin)


pfr  export -c inputs/cfpa_mcxn9xx.yaml -o workspace/cfpa.bin 
Success. (CFPA binary has been generated and stored into workspace/cfpa.bin)


In [8]:
import os
import time

DEVHSM_CONFIG = INPUTS + "devhsm_config.yaml"
DEVHSM_SB = WORKSPACE + "mcxn9xx_devhsm.sb"

# generate the provisioning SB file on the device (CUST_MK_SK = pck.txt, CMPA, CFPA)
%! nxpdevhsm $VERBOSITY generate $USB -c $DEVHSM_CONFIG
assert os.path.exists(DEVHSM_SB)

# apply the provisioning SB file and reset the device to load the new configuration
%! blhost $USB receive-sb-file $DEVHSM_SB
%! blhost $USB reset
# wait for the device to boot again after the reset
time.sleep(3)

nxpdevhsm  generate -u 0x1fc9,0x014f -c inputs/devhsm_config.yaml 
 1: Initial target reset is disabled 
 2: Generating OEM master share.
 3: Generating 48 bytes FW signing keys.
 4: Generating 48 bytes FW encryption keys.
 5: Creating template un-encrypted SB3 header and data blobs.
 5.1: Creating template SB3 header.
 5.2: Creating un-encrypted SB3 data.
 5.3 Looking for plaintext keys to wrap
 5.4 Handling CUST_MK_SK/SBKEK keyblob
 5.4.1 Injecting wrapped key
 5.4.2: Wrapping CUST_MK_SK key.
 5.4.3: Adding wrapped CUST_MK_SK LoadKeyBlob command into SB file.
 6: Encrypting SB3 data on device
 6.1: Enriching encrypted SB3 data by mandatory hashes.
 6.2: Creating dummy certificate block.
 6.3: Updating SB3 header by valid values.
 6.4: Preparing SB3 manifest to sign.
 7: Creating SB3 manifest signature on device.
 8: Composing final SB3 file.
 9: Resetting the target device
Final SB file has been written: workspace/mcxn9xx_devhsm.sb


blhost -u 0x1fc9,0x014f receive-sb-file workspace/mcxn9xx_devhsm.sb 
Response status = 0 (0x0) Success.


blhost -u 0x1fc9,0x014f reset 
Response status = 0 (0x0) Success.


## 9. Send the SB3.1 file to the device

The last step is to upload the SB3.1 file to the device. The `blhost receive-sb-file` command sends the secure binary to the target, which authenticates it against the Root of Trust, decrypts it with the provisioned Part Common Key, and processes the contained commands.

In [9]:
# uploads the SB3.1 file
SB31_FILE = WORKSPACE + "mcxn9xx_sb31.sb3"
%! blhost $USB receive-sb-file $SB31_FILE

blhost -u 0x1fc9,0x014f receive-sb-file workspace/mcxn9xx_sb31.sb3 
Response status = 0 (0x0) Success.
